In [3]:
# %%
# =============================================================================
# 05_deepeval_validation.ipynb
# Financial AI Governance — DeepEval Validity Verification
# Kernel : Python (deepeval_env)
# Input  : results/responses/responses_baseline.json
#          results/responses/responses_rag.json
#          results/scores/scores_all.csv
# Output : results/scores/scores_deepeval.csv
#          results/tables/table9_deepeval_validity.csv
# =============================================================================

# %%
# =============================================================================
# Cell 1. Libraries and Environment Setup
# =============================================================================
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from dotenv import load_dotenv

# Disable DeepEval progress bar widgets (prevents "model not found" widget errors)
os.environ['DEEPEVAL_TELEMETRY_OPT_OUT'] = 'YES'
os.environ['DEEPEVAL_UPDATE_WARNING']     = 'NO'

import deepeval

from deepeval import evaluate
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric,
)
from deepeval.test_case import LLMTestCase

# Suppress tqdm-based widget rendering in Jupyter
try:
    from deepeval.utils import set_should_use_rich_print
    set_should_use_rich_print(False)
except Exception:
    pass

# Directory paths
RESPONSE_DIR = '../results/responses'
SCORE_DIR    = '../results/scores'
TABLE_DIR    = '../results/tables'

for d in [SCORE_DIR, TABLE_DIR]:
    os.makedirs(d, exist_ok=True)

# API setup
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
EVAL_MODEL     = os.getenv('LLM_MODEL', 'gpt-4o-mini')

if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY not set in .env")

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

print(f"[INFO] DeepEval evaluation model : {EVAL_MODEL}")
print(f"[INFO] SCORE_DIR : {os.path.abspath(SCORE_DIR)}")


# %%
# =============================================================================
# Cell 2. Load Data
# =============================================================================
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

baseline = load_json(os.path.join(RESPONSE_DIR, 'responses_baseline.json'))
rag      = load_json(os.path.join(RESPONSE_DIR, 'responses_rag.json'))

df_bl       = pd.DataFrame(baseline)
df_rag      = pd.DataFrame(rag)
df_g_scores = pd.read_csv(os.path.join(SCORE_DIR, 'scores_all.csv'))

print(f"[INFO] Baseline records  : {len(df_bl)}")
print(f"[INFO] RAG records       : {len(df_rag)}")
print(f"[INFO] G1~G4 score rows  : {len(df_g_scores)}")


# %%
# =============================================================================
# Cell 3. DeepEval Metrics Configuration
# =============================================================================
# Three DeepEval metrics for cross-validation with G1~G4:
#
# AnswerRelevancy  → validates G1 (Accuracy): is the response relevant to the question?
# Faithfulness     → validates G3 (Transparency): is the response faithful to the context?
#                    (RAG only — requires retrieval context)
# Hallucination    → validates G1+G3: does the response contain hallucinated facts?
#                    (RAG only — requires retrieval context)
#
# Mapping to paper:
#   DeepEval AnswerRelevancy ↔ G1 Accuracy    (both conditions)
#   DeepEval Faithfulness    ↔ G3 Transparency (RAG only)
#   DeepEval Hallucination   ↔ G1+G3 combined (RAG only)

THRESHOLD = 0.5   # DeepEval pass/fail threshold

metric_answer_relevancy = AnswerRelevancyMetric(
    threshold    = THRESHOLD,
    model        = EVAL_MODEL,
    async_mode   = False,
    verbose_mode = False,
)
metric_faithfulness = FaithfulnessMetric(
    threshold    = THRESHOLD,
    model        = EVAL_MODEL,
    async_mode   = False,
    verbose_mode = False,
)
metric_hallucination = HallucinationMetric(
    threshold    = THRESHOLD,
    model        = EVAL_MODEL,
    async_mode   = False,
    verbose_mode = False,
)

print("[INFO] DeepEval metrics initialized")
print(f"  AnswerRelevancy  → validates G1 Accuracy    (threshold={THRESHOLD})")
print(f"  Faithfulness     → validates G3 Transparency (threshold={THRESHOLD}, RAG only)")
print(f"  Hallucination    → validates G1+G3 combined  (threshold={THRESHOLD}, RAG only)")


# %%
# =============================================================================
# Cell 4. Sampling Strategy
# =============================================================================
# Full 600-record evaluation is expensive.
# Stratified sample: 10 per regulation × 2 conditions × 3 difficulty levels
# = up to 180 records for DeepEval cross-validation.
# This is sufficient for validity verification in DSR methodology.

np.random.seed(42)

SAMPLE_PER_STRATUM = 10   # per regulation × difficulty stratum

def stratified_sample(df: pd.DataFrame, n: int) -> pd.DataFrame:
    """Sample n records per regulation × difficulty stratum."""
    groups = df.groupby(['regulation', 'difficulty'])
    sampled = []
    for _, grp in groups:
        s = grp.sample(min(n, len(grp)), random_state=42)
        sampled.append(s)
    return pd.concat(sampled).reset_index(drop=True)

sample_bl  = stratified_sample(df_bl,  SAMPLE_PER_STRATUM)
sample_rag = stratified_sample(df_rag, SAMPLE_PER_STRATUM)

print(f"[INFO] Stratified sample — Baseline : {len(sample_bl)} records")
print(f"[INFO] Stratified sample — RAG      : {len(sample_rag)} records")
print(f"[INFO] Total DeepEval evaluations   : {len(sample_bl) + len(sample_rag)}")

# Distribution check
print("\n[Sample Distribution — Regulation × Difficulty]")
print(sample_rag.groupby(['regulation','difficulty']).size().unstack(fill_value=0).to_string())


# %%
# =============================================================================
# Cell 5. Run DeepEval — Baseline (AnswerRelevancy only)
# =============================================================================
# Baseline has no retrieval context → only AnswerRelevancy applicable

print("[RUN] DeepEval — Baseline (AnswerRelevancy) ...")
print(f"      Records: {len(sample_bl)}\n")

bl_results = []

for i, row in sample_bl.iterrows():
    test_case = LLMTestCase(
        input            = row['question'],
        actual_output    = row['response'],
        expected_output  = row['ground_truth'],
        retrieval_context= [],   # no context for baseline
    )
    try:
        metric_answer_relevancy.measure(test_case)
        ar_score = metric_answer_relevancy.score
        ar_pass  = metric_answer_relevancy.success
    except Exception as e:
        ar_score = -1.0
        ar_pass  = False
        print(f"  [WARN] AnswerRelevancy error at {row['id']}: {e}")

    bl_results.append({
        'id'               : row['id'],
        'condition'        : 'baseline',
        'regulation'       : row['regulation'],
        'difficulty'       : row['difficulty'],
        'financial_domain' : row['financial_domain'],
        'governance_axis'  : row['governance_axis'],
        'answer_relevancy' : round(ar_score, 4),
        'ar_pass'          : ar_pass,
        'faithfulness'     : None,   # N/A for baseline
        'faith_pass'       : None,
        'hallucination'    : None,   # N/A for baseline
        'hall_pass'        : None,
    })

    if (len(bl_results)) % 20 == 0:
        valid  = [r for r in bl_results if r['answer_relevancy'] >= 0]
        avg_ar = np.mean([r['answer_relevancy'] for r in valid]) if valid else 0
        print(f"  [Checkpoint {len(bl_results):3d}/{len(sample_bl)}] "
              f"avg AnswerRelevancy: {avg_ar:.3f}")

print(f"\n[INFO] Baseline DeepEval complete.")


# %%
# =============================================================================
# Cell 6. Run DeepEval — RAG (All Three Metrics)
# =============================================================================
# RAG has retrieval context → all three metrics applicable

print("[RUN] DeepEval — RAG (AnswerRelevancy + Faithfulness + Hallucination) ...")
print(f"      Records: {len(sample_rag)}\n")

rag_results = []

for i, row in sample_rag.iterrows():
    # Parse context: split on separator used in Notebook 03
    ctx_chunks = row['context_used'].split('\n\n---\n\n') \
                 if pd.notna(row['context_used']) and row['context_used'] else []

    test_case = LLMTestCase(
        input            = row['question'],
        actual_output    = row['response'],
        expected_output  = row['ground_truth'],
        retrieval_context= ctx_chunks,
    )

    # AnswerRelevancy
    try:
        metric_answer_relevancy.measure(test_case)
        ar_score = metric_answer_relevancy.score
        ar_pass  = metric_answer_relevancy.success
    except Exception as e:
        ar_score, ar_pass = -1.0, False

    # Faithfulness
    try:
        metric_faithfulness.measure(test_case)
        faith_score = metric_faithfulness.score
        faith_pass  = metric_faithfulness.success
    except Exception as e:
        faith_score, faith_pass = -1.0, False

    # Hallucination
    try:
        metric_hallucination.measure(test_case)
        hall_score = metric_hallucination.score
        hall_pass  = metric_hallucination.success
    except Exception as e:
        hall_score, hall_pass = -1.0, False

    rag_results.append({
        'id'               : row['id'],
        'condition'        : 'rag',
        'regulation'       : row['regulation'],
        'difficulty'       : row['difficulty'],
        'financial_domain' : row['financial_domain'],
        'governance_axis'  : row['governance_axis'],
        'answer_relevancy' : round(ar_score, 4),
        'ar_pass'          : ar_pass,
        'faithfulness'     : round(faith_score, 4),
        'faith_pass'       : faith_pass,
        'hallucination'    : round(hall_score, 4),
        'hall_pass'        : hall_pass,
    })

    if len(rag_results) % 20 == 0:
        valid_ar   = [r for r in rag_results if r['answer_relevancy'] >= 0]
        valid_fa   = [r for r in rag_results if r['faithfulness'] is not None and r['faithfulness'] >= 0]
        avg_ar     = np.mean([r['answer_relevancy'] for r in valid_ar]) if valid_ar else 0
        avg_faith  = np.mean([r['faithfulness'] for r in valid_fa]) if valid_fa else 0
        print(f"  [Checkpoint {len(rag_results):3d}/{len(sample_rag)}] "
              f"AnswerRelevancy: {avg_ar:.3f} | Faithfulness: {avg_faith:.3f}")

print(f"\n[INFO] RAG DeepEval complete.")


# %%
# =============================================================================
# Cell 7. Save DeepEval Scores
# =============================================================================
df_deepeval = pd.DataFrame(bl_results + rag_results)

out_de = os.path.join(SCORE_DIR, 'scores_deepeval.csv')
df_deepeval.to_csv(out_de, index=False, encoding='utf-8-sig')
print(f"[SAVE] {out_de}")
print(f"[INFO] Total records saved: {len(df_deepeval)}")


# %%
# =============================================================================
# Cell 8. Cross-Validation: DeepEval ↔ G1~G4 Correlation
# =============================================================================
# Merge DeepEval scores with G1~G4 scores on id + condition
df_g = pd.read_csv(os.path.join(SCORE_DIR, 'scores_all.csv'))

# Merge RAG subset only (faithfulness and hallucination available)
df_rag_de = df_deepeval[df_deepeval['condition'] == 'rag'].copy()
df_rag_g  = df_g[df_g['condition'] == 'rag'][
    ['id', 'G1_score', 'G2_score', 'G3_score', 'G4_score', 'governance_score']
]

df_merged = df_rag_de.merge(df_rag_g, on='id', how='inner')

print(f"[INFO] Merged records for correlation: {len(df_merged)}")
print("\n[DeepEval ↔ G-Axis Pearson Correlation]")

corr_pairs = [
    ('answer_relevancy', 'G1_score',        'AnswerRelevancy ↔ G1 Accuracy'),
    ('faithfulness',     'G3_score',        'Faithfulness    ↔ G3 Transparency'),
    ('hallucination',    'G1_score',        'Hallucination   ↔ G1 Accuracy'),
    ('answer_relevancy', 'governance_score','AnswerRelevancy ↔ Governance Score'),
    ('faithfulness',     'governance_score','Faithfulness    ↔ Governance Score'),
]

corr_rows = []
for de_col, g_col, label in corr_pairs:
    sub = df_merged[[de_col, g_col]].dropna()
    # Skip if insufficient variance (all same value → NaN correlation)
    if len(sub) < 5 or sub[de_col].std() == 0:
        corr_rows.append({'Metric Pair': label, 'Pearson r': 'N/A (no variance)', 'N': len(sub)})
        print(f"  {label:45s} : N/A (insufficient variance, N={len(sub)})")
        continue
    r = sub[de_col].corr(sub[g_col])
    corr_rows.append({'Metric Pair': label, 'Pearson r': round(r, 4), 'N': len(sub)})
    print(f"  {label:45s} : r = {r:.4f}  (N={len(sub)})")

df_corr = pd.DataFrame(corr_rows)


# %%
# =============================================================================
# Cell 9. Table 9 — DeepEval Validity Summary (for Paper)
# =============================================================================
REG_MAP = {
    'NIST_AI_RMF'    : 'NIST AI RMF',
    'KR_AI_BASIC_ACT': 'Korean AI Basic Act',
    'EU_AI_ACT'      : 'EU AI Act',
}

rows = []

# Baseline — AnswerRelevancy only
for reg_key, reg_label in REG_MAP.items():
    sub = df_deepeval[
        (df_deepeval['condition'] == 'baseline') &
        (df_deepeval['regulation'] == reg_key)
    ]
    valid = sub[sub['answer_relevancy'] >= 0]
    if len(valid) == 0:
        continue
    rows.append({
        'Condition'        : 'Baseline',
        'Regulation'       : reg_label,
        'N'                : len(valid),
        'AnswerRelevancy'  : round(valid['answer_relevancy'].mean(), 3),
        'Pass Rate AR (%)'  : round(valid['ar_pass'].mean() * 100, 1),
        'Faithfulness'     : 'N/A',
        'Pass Rate FA (%)' : 'N/A',
        'Hallucination'    : 'N/A',
        'Pass Rate HA (%)' : 'N/A',
    })

# RAG — All three metrics
for reg_key, reg_label in REG_MAP.items():
    sub = df_deepeval[
        (df_deepeval['condition'] == 'rag') &
        (df_deepeval['regulation'] == reg_key)
    ]
    # Filter only rows where AnswerRelevancy is valid
    # Faithfulness/Hallucination treated separately (may be NaN)
    valid = sub[sub['answer_relevancy'] >= 0]
    if len(valid) == 0:
        continue

    faith_valid = valid[valid['faithfulness'].notna() & (valid['faithfulness'] >= 0)]
    hall_valid  = valid[valid['hallucination'].notna() & (valid['hallucination'] >= 0)]

    rows.append({
        'Condition'        : 'RAG',
        'Regulation'       : reg_label,
        'N'                : len(valid),
        'AnswerRelevancy'  : round(valid['answer_relevancy'].mean(), 3),
        'Pass Rate AR (%)' : round(valid['ar_pass'].mean() * 100, 1),
        'Faithfulness'     : round(faith_valid['faithfulness'].mean(), 3) if len(faith_valid) > 0 else 'N/A',
        'Pass Rate FA (%)' : round(faith_valid['faith_pass'].mean() * 100, 1) if len(faith_valid) > 0 else 'N/A',
        'Hallucination'    : round(hall_valid['hallucination'].mean(), 3) if len(hall_valid) > 0 else 'N/A',
        'Pass Rate HA (%)' : round(hall_valid['hall_pass'].mean() * 100, 1) if len(hall_valid) > 0 else 'N/A',
    })

table9 = pd.DataFrame(rows)
out_t9 = os.path.join(TABLE_DIR, 'table9_deepeval_validity.csv')
table9.to_csv(out_t9, index=False, encoding='utf-8-sig')

print("[Table 9] DeepEval Validity Summary")
print(table9.to_string(index=False))
print(f"\n[SAVE] {out_t9}")

# Append correlation table
out_corr = os.path.join(TABLE_DIR, 'table9b_deepeval_correlation.csv')
df_corr.to_csv(out_corr, index=False, encoding='utf-8-sig')
print(f"[SAVE] {out_corr}")

print(f"\n✅ Notebook 05 complete — Next: 06_llm_as_judge.ipynb")

[INFO] DeepEval evaluation model : gpt-4o-mini
[INFO] SCORE_DIR : C:\Users\User\Downloads\학술\9_ai거버넌스_챗봇_ssci\financial_ai_governance\results\scores
[INFO] Baseline records  : 300
[INFO] RAG records       : 300
[INFO] G1~G4 score rows  : 600
[INFO] DeepEval metrics initialized
  AnswerRelevancy  → validates G1 Accuracy    (threshold=0.5)
  Faithfulness     → validates G3 Transparency (threshold=0.5, RAG only)
  Hallucination    → validates G1+G3 combined  (threshold=0.5, RAG only)
[INFO] Stratified sample — Baseline : 90 records
[INFO] Stratified sample — RAG      : 90 records
[INFO] Total DeepEval evaluations   : 180

[Sample Distribution — Regulation × Difficulty]
difficulty       advanced  basic  intermediate
regulation                                    
EU_AI_ACT              10     10            10
KR_AI_BASIC_ACT        10     10            10
NIST_AI_RMF            10     10            10
[RUN] DeepEval — Baseline (AnswerRelevancy) ...
      Records: 90



Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()


[INFO] RAG DeepEval complete.
[SAVE] ../results/scores\scores_deepeval.csv
[INFO] Total records saved: 180
[INFO] Merged records for correlation: 90

[DeepEval ↔ G-Axis Pearson Correlation]
  AnswerRelevancy ↔ G1 Accuracy                 : r = 0.1057  (N=90)
  Faithfulness    ↔ G3 Transparency             : r = -0.1324  (N=90)
  Hallucination   ↔ G1 Accuracy                 : N/A (insufficient variance, N=90)
  AnswerRelevancy ↔ Governance Score            : r = 0.0332  (N=90)
  Faithfulness    ↔ Governance Score            : r = -0.0593  (N=90)
[Table 9] DeepEval Validity Summary
Condition          Regulation  N  AnswerRelevancy  Pass Rate AR (%) Faithfulness Pass Rate FA (%) Hallucination Pass Rate HA (%)
 Baseline         NIST AI RMF 30            0.943             100.0          N/A              N/A           N/A              N/A
 Baseline Korean AI Basic Act 30            0.972             100.0          N/A              N/A           N/A              N/A
 Baseline           EU A